In [43]:
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchText
from openai import AzureOpenAI
import os
from dotenv import load_dotenv

In [31]:
load_dotenv()
COLLECTION_NAME = "recipe_rag_collection"
URL_QDRANT = "https://d98e588d-5738-4671-ba4a-32e50e71f68d.europe-west3-0.gcp.cloud.qdrant.io:6333"
API_KEY_QDRANT = os.getenv("API_KEY_QDRANT")
ENDPOINT_AZURE = "https://rag-recipe-resource.openai.azure.com/"
LLM_MODEL_AZURE = "gpt-35-turbo"
EMBEDDING_MODEL_AZURE = "text-embedding-3-large"
API_VERSION_AZURE = "2024-12-01-preview"
API_KEY_AZURE = os.getenv('API_KEY_AZURE')

In [32]:
client_qdrant = QdrantClient(
    url=URL_QDRANT,
    api_key=API_KEY_QDRANT
)

In [33]:
client_azure = AzureOpenAI(
    azure_endpoint=ENDPOINT_AZURE,
    api_key=API_KEY_AZURE,
    api_version=API_VERSION_AZURE
)

In [34]:
def response_generator(system_prompt, user_prompt):
    response = client_azure.chat.completions.create(
        model="gpt-35-turbo",
        messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
    )

    return response.choices[0].message.content

In [35]:
def recommend_recipes(ingredients, top_k=10):
    query_emb = client_azure.embeddings.create(
        input=ingredients,
        model=EMBEDDING_MODEL_AZURE
    ).data[0].embedding

    results = client_qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_emb,
        limit=top_k
    )

    retrieved_contexts = [point.payload["title"] for point in results.points]
    context = "\n".join(retrieved_contexts)

    system_prompt = """
        [SYSTEM INSTRUCTION]
        You are a highly detailed and practical **Chef AI**. Your goal is to generate a a list of
         dishes from context.
        Provide encouraging tone.
    """
    user_prompt = f"""
        [TASK DEFINITION]
        Task: Given the dish's title, **generate a complete list of dishes**. If the answer is not contained in the context, say "I don't know.
        
        [CONTEXT/INPUT BLOCK]
        --- DISH TITLE ---
        {context}
        ---
        
        [OUTPUT CONSTRAINTS AND FORMAT]
        Constraints:
        1. Do not include any introductory or concluding sentences. Start directly with the title.
        2. Use Markdown formatting for headings and lists.
        
        Format:
        # Dishes suggestions, according to available ingredients:
        * [Dish Title 1]
        * [Dish Title 2]
        * ... 
    """

    return response_generator(system_prompt, user_prompt)



In [44]:
def query_by_title(title):
    search_filter = Filter(
        must=[
            FieldCondition(
                key="title",
                match=MatchText(text=title)
            )
        ]
    )

    search_result, _ = client_qdrant.scroll(
        collection_name=COLLECTION_NAME,
        scroll_filter=search_filter,
        limit=1,
        with_payload=True
    )

    if search_result:
        return search_result[0].payload

In [37]:
def cooking_instructions(title):
    directions = query_by_title(title)['directions']

    system_prompt = """
        You are a highly detailed and practical **Chef AI**. Your goal is to generate a comprehensive,
        easy-to-follow cooking guide for a specific dish, designed for an average home cook.
        Provide precise details and an encouraging tone.
        """
    user_prompt = f"""
        [TASK DEFINITION]
        Task: Given the dish's title, **generate a complete recipe**. This must include a detailed
        **step-by-step cooking instructions**. If the answer is not contained in the context, say "I don't know.

        [CONTEXT/INPUT BLOCK]
        --- DISH TITLE ---
        {title}
        --- INSTRUCTIONS ---
        {directions}

        [OUTPUT CONSTRAINTS AND FORMAT]
        Constraints:
        1. Ensure all measurements are clear (e.g., "1 cup," "2 tsp," "300g").
        2. Do not include any introductory or concluding sentences. Start directly with the title.
        3. Use Markdown formatting for headings and lists.

        Format:
        # [Dish Title]

        ## Instructions
        1. [Step 1]
        2. [Step 2]
        3. ...
    """

    return response_generator(system_prompt, user_prompt)

In [38]:
def required_ingredients(title):
    ingredients = query_by_title(title)['ingredients']

    system_prompt = """
        [SYSTEM INSTRUCTION]
        You are a highly detailed and practical **Chef AI**. Your goal is to generate a comprehensive,
        list of ingredients for a specific dish, designed for an average home cook.
        Provide precise details and an encouraging tone.
    """
    user_prompt = f"""
        [TASK DEFINITION]
        Task: Given the dish's title, **generate a complete list of ingedients**. This must include a **full ingredients list** with specific measurements. If the answer is not contained in the context, say "I don't know.

        [CONTEXT/INPUT BLOCK]
        --- DISH TITLE ---
        {title}
        --- INGREDIENTS ---
        {ingredients}

        [OUTPUT CONSTRAINTS AND FORMAT]
        Constraints:
        1. Ensure all measurements are clear (e.g., "1 cup," "2 tsp," "300g").
        2. Do not include any introductory or concluding sentences. Start directly with the title.
        3. Use Markdown formatting for headings and lists.

        Format:
        # [Dish aTitle]

        ## Ingredients
        * [Ingredient 1]: [Amount]
        * [Ingredient 2]: [Amount]
        * ...
    """

    return response_generator(system_prompt, user_prompt)

In [39]:
dishes = recommend_recipes(['chicken', 'flour', 'cheese', 'vinegar', 'milk', 'butter'])

In [40]:
print(dishes)

# Dishes suggestions, according to available ingredients:
* Party Chicken
* Cheese Chicken
* Chinese Chicken
* Great Chicken
* Rotisserie Chicken
* Crispy Chip Chicken
* Mexican Chicken
* Lazy Chicken
* He-Man Chicken


In [47]:
instruction = cooking_instructions("Mexican Chicken")

In [48]:
print(instruction)


# Mexican Chicken

## Instructions
1. Heat 1 tablespoon of oil and 1 tablespoon of butter in a large skillet over medium heat until the butter melts.
2. Add 1 pound of chicken pieces, 1 chopped onion, 1 chopped bell pepper, and your preferred seasonings to the skillet. Cook until the chicken is fully cooked and the vegetables are soft.
3. Stir in 1 cup of rice, 2 cups of chicken stock, and 1 can of cream of chicken soup. Simmer over low heat until the rice is tender, stirring occasionally.
4. Serve and enjoy! This flavorful Mexican chicken mixture can also be used as a delicious filling for burritos or enchiladas.


In [49]:
ingredients = required_ingredients("Mexican Chicken")

In [50]:
print(ingredients)


# Mexican Chicken

## Ingredients
* 1 tablespoon vegetable oil
* 1 tablespoon butter
* 1 lb boneless skinless chicken breast, cut into bite-sized chunks
* 1/2 cup onion, finely chopped
* 1 poblano pepper, finely chopped
* 1 cup instant rice
* 1 cup chicken stock (or broth)
* 1/2 teaspoon chili powder
* 1/4 teaspoon pepper
* 1/4 teaspoon paprika
* 1 (10 ounce) can cream of mushroom soup
